# Generative AI Optimized Recommendations (boto3)

This notebook uses the **SageMaker AI Recommendation Service** to automatically find the optimal serving configuration for a generative AI model. The service analyzes your model and workload characteristics to generate deployment recommendations optimized for cost, latency, or throughput — evaluating instance types and applying optimizations like speculative decoding and kernel tuning.

**Workflow:**
1. (Optional) Download model weights and upload to S3
2. Create a workload configuration defining your traffic patterns
3. Submit a recommendation job pointing to your model in S3
4. Poll until the job completes
5. Review the recommended deployment configuration and expected performance
6. Clean up resources

**Prerequisites:**
- Model weights in S3 in HuggingFace SafeTensor format
- An IAM execution role with trust policy allowing `sagemaker.amazonaws.com` and SageMaker + S3 permissions
- Caller IAM identity with `sagemaker:*` permissions
- An S3 output bucket for recommendation artifacts

## Step 1 — Install dependencies

Upgrade boto3 to ensure the latest API features are available.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

## Step 2 — Configuration

**You must provide your own values below before running this notebook:**
- `MODEL_S3_URI` — S3 path to your model weights in HuggingFace SafeTensor format (e.g. `s3://your-bucket/models/gpt-oss-20b/`)
- `S3_OUTPUT` — S3 path where recommendation outputs will be written (e.g. `s3://your-bucket/recommendations/`)

The example below uses `openai/gpt-oss-20b` as the model. If you don't have it in S3 yet, see the **optional download cell** in Step 2b.

> **Note:** `ROLE_ARN` is automatically retrieved from the SageMaker execution role attached to this Studio environment via `sagemaker.get_execution_role()`.

In [ ]:
import json
import time
import boto3
from datetime import datetime
import uuid

def get_execution_role():
    sts = boto3.client('sts')
    arn = sts.get_caller_identity()['Arn']
    if ':assumed-role/' in arn:
        parts = arn.split(':')
        account = parts[4]
        role_name = parts[5].split('/')[1]
        return f"arn:aws:iam::{account}:role/{role_name}"
    raise RuntimeError(f"Cannot derive execution role from ARN: {arn}. Set ROLE_ARN manually.")

REGION       = "us-west-2"  # <-- change if deploying in a different region
MODEL_S3_URI = "s3://your-bucket/models/gpt-oss-20b/"       # <-- replace with your S3 path
S3_OUTPUT    = "s3://your-bucket/recommendations/"          # <-- replace with your S3 output path

# Auto-generate unique names on every run
run_id      = datetime.now().strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
CONFIG_NAME = f"rec-config-{run_id}"
JOB_NAME    = f"rec-job-{run_id}"

session = boto3.Session(region_name=REGION)
sm = session.client("sagemaker")
ROLE_ARN = get_execution_role()

print(f"Role ARN    : {ROLE_ARN}")
print(f"Config name : {CONFIG_NAME}")
print(f"Job name    : {JOB_NAME}")

### Ensure role trust policy allows SageMaker

Checks the trust policy of the execution role and adds `sagemaker.amazonaws.com` if missing. This is a no-op if the trust policy is already correct.

In [ ]:
iam = boto3.client('iam')
role_name = ROLE_ARN.split('/')[-1]

trust = iam.get_role(RoleName=role_name)['Role']['AssumeRolePolicyDocument']
principals = [
    s.get('Principal', {}).get('Service', '')
    for s in trust.get('Statement', [])
]
flat = [p for item in principals for p in (item if isinstance(item, list) else [item])]

if 'sagemaker.amazonaws.com' in flat:
    print('✅ Trust policy already allows sagemaker.amazonaws.com')
else:
    trust['Statement'].append({
        'Effect': 'Allow',
        'Principal': {'Service': 'sagemaker.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    })
    iam.update_assume_role_policy(
        RoleName=role_name,
        PolicyDocument=json.dumps(trust)
    )
    print(f'✅ Added sagemaker.amazonaws.com to trust policy for role: {role_name}')

## Step 2b — (Optional) Download openai/gpt-oss-20b and upload to S3

Skip this step if your model weights are already in S3.

This cell downloads `openai/gpt-oss-20b` from Hugging Face using `hf_transfer` for fast multi-part download, then syncs the weights to your S3 bucket. Update `MODEL_S3_URI` in Step 2 to match your bucket before running.

In [ ]:
# Optional: download gpt-oss-20b from Hugging Face and upload to S3
# Uncomment and run this cell if you need to upload the model to S3 first.
#
# LOCAL_MODEL_DIR = "/tmp/gpt-oss-20b"
#
# !pip install -q "huggingface_hub[hf_transfer]"
# !HF_HUB_ENABLE_HF_TRANSFER=1 huggingface-cli download openai/gpt-oss-20b \
#     --local-dir {LOCAL_MODEL_DIR} \
#     --exclude '*.gguf' '*.bin'
#
# !aws s3 sync {LOCAL_MODEL_DIR} {MODEL_S3_URI}
# print(f"Model uploaded to {MODEL_S3_URI}")

In [ ]:
# Uncomment and run this cell if you need to upload the model to S3 first.
#
# LOCAL_MODEL_DIR = "/tmp/gpt-oss-20b"
#
# !pip install -q "huggingface_hub[hf_transfer]"
# !HF_HUB_ENABLE_HF_TRANSFER=1 huggingface-cli download openai/gpt-oss-20b \
#     --local-dir /tmp/gpt-oss-20b \
#     --exclude '*.gguf' '*.bin'
#
# !aws s3 sync /tmp/gpt-oss-20b {MODEL_S3_URI}
# print(f"Model uploaded to {MODEL_S3_URI}")

## Step 3 — Create workload config

A workload config defines the traffic pattern the recommendation service uses when evaluating candidate configurations. It tells the service what your real workload looks like so recommendations are tailored to your use case.

**Dataset options:**
| Source | How to configure |
|---|---|
| Synthetic (default) | Set `prompt_input_tokens_mean` / `output_tokens_mean` — no dataset needed |
| Public dataset (e.g. ShareGPT) | Add `"public_dataset": "sharegpt"` to `parameters` |
| Custom dataset from S3 | Add `custom_dataset_type` + `input_file` to `parameters` and a `DatasetConfig` |

This example uses the ShareGPT public dataset with a concurrency of 5 and 300-second duration.

In [ ]:
# 1. Create workload config
workload_spec = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "concurrency": 5,
        "public_dataset": "sharegpt",    # optional, will use synth dataset if omitted
        "prompt_input_tokens_mean": 500,
        "prompt_input_tokens_stddev": 10,
        "output_tokens_mean": 500,
        "output_tokens_stddev": 10,
    },
}

config_resp = sm.create_ai_workload_config(
    AIWorkloadConfigName=CONFIG_NAME,
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(workload_spec)}},
)
print(f"Config created: {config_resp['AIWorkloadConfigArn']}")

## Step 4 — Submit recommendation job

The recommendation job analyzes your model and generates deployment recommendations. Key parameters:

| Parameter | Description |
|---|---|
| `ModelSource` | S3 URI of your model weights in HuggingFace SafeTensor format |
| `PerformanceTarget` | Metric to optimize — `throughput`, `ttft-ms`, or `request-latency-ms` |
| `ComputeSpec` | Instance types to evaluate (e.g. `ml.g6.24xlarge`) |
| `OptimizeModel` | `False` = config search only; `True` = also apply optimizations like speculative decoding |
| `InferenceSpecification` | Serving framework — `VLLM` or `LMI` |

The service outputs a `ModelPackage` with `InferenceSpecifications` for each recommended instance, along with `ExpectedPerformance` metrics (latency, throughput, cost).

In [ ]:
# 2. Create recommendation job
job_resp = sm.create_ai_recommendation_job(
    AIRecommendationJobName=JOB_NAME,
    ModelSource={
        "S3": {
            "S3Uri": MODEL_S3_URI,
        }
    },
    OutputConfig={"S3OutputLocation": S3_OUTPUT},
    AIWorkloadConfigIdentifier=CONFIG_NAME,
    RoleArn=ROLE_ARN,
    PerformanceTarget={
        "Constraints": [
            {"Metric": "throughput"},
        ]
    },
    ComputeSpec={
        "InstanceTypes": ["ml.g6.24xlarge"]
    },
    OptimizeModel=False,
    InferenceSpecification = {"Framework": "VLLM"},
)
print(f"Job created: {job_resp['AIRecommendationJobArn']}")

## Step 5 — Poll for completion

Polls every 30 seconds until the job reaches a terminal state: `Completed`, `Failed`, or `Stopped`.

In [ ]:
# 3. Poll until terminal
while True:
    desc = sm.describe_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)
    status = desc["AIRecommendationJobStatus"]
    print(f"  Status: {status}")
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(30)

## Step 6 — Review recommendations

Each recommendation in the response includes:
- **`OptimizationDetails`** — optimizations applied (e.g. `SpeculativeDecoding`, `Quantization`, `KernelTuning`)
- **`ModelDetails`** — `ModelPackageArn` and `InferenceSpecificationName` to use when deploying
- **`DeploymentConfiguration`** — container image, instance type, environment variables (tensor parallelism, max sequences, speculative config)
- **`ExpectedPerformance`** — predicted latency (p50/p90/p99), throughput (tokens/s), and TTFT

To deploy a recommendation, pass the `ModelPackageArn` and `InferenceSpecificationName` to `CreateModel`.

In [ ]:
# 4. Print recommendations
if status == "Completed":
    recommendations = desc.get("Recommendations", [])
    print(f"\n{len(recommendations)} recommendation(s) found:")
    for i, rec in enumerate(recommendations, 1):
        print(f"\n--- Recommendation {i} ---")
        print(f"  {rec.get('RecommendationDescription', 'N/A')}")
        if rec.get("InstanceDetails"):
            for inst in rec["InstanceDetails"]:
                print(f"  Instance: {inst['InstanceType']} x{inst.get('InstanceCount', 1)}")
        if rec.get("OptimizationDetails"):
            for opt in rec["OptimizationDetails"]:
                print(f"  Optimization: {opt['OptimizationType']}")
        if rec.get("ExpectedPerformance"):
            for perf in rec["ExpectedPerformance"]:
                print(f"  {perf['Metric']}: {perf['Value']} {perf.get('Unit', '')}")
        if rec.get("DeploymentConfiguration"):
            deploy = rec["DeploymentConfiguration"]
            print(f"  Deploy image: {deploy.get('ImageUri', 'N/A')}")
            print(f"  Deploy instance: {deploy.get('InstanceType', 'N/A')}")
else:
    print(f"\nEnded with status: {status}")
    if "FailureReason" in desc:
        print(f"Reason: {desc['FailureReason']}")

In [ ]:
print(json.dumps(desc.get("Recommendations", []), indent=2))

## Step 7 — Cleanup

Delete the recommendation job and workload config to avoid leaving unused resources in your account.

> **Note:** A job must be in a terminal state (`Completed`, `Failed`, or `Stopped`) before it can be deleted. Call `stop_ai_recommendation_job` first if the job is still running.

In [ ]:
sm.stop_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)

In [ ]:
sm.delete_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)

In [ ]:
sm.delete_ai_workload_config(AIWorkloadConfigName=CONFIG_NAME)